# COMPASS → SQLite
Reads all COMPASS pipeline output files and loads them into a single SQLite database with `sample_id` (GCA accession) as the shared foreign key.

In [ ]:
# ── CONFIGURE THESE ────────────────────────────────────────────────────────
RESULTS_DIR = "/fastscratch/tylerdoe/COMPASS-runs/new_set/results_ecoli_8k"
DB_PATH     = "/fastscratch/tylerdoe/COMPASS-runs/new_set/compass_ecoli_8k.db"
# ───────────────────────────────────────────────────────────────────────────

In [ ]:
import sqlite3
import re
from pathlib import Path
from collections import Counter

import pandas as pd

results = Path(RESULTS_DIR)
con = sqlite3.connect(DB_PATH)
print(f"Connected to {DB_PATH}")

## Assembly quality — QUAST

In [ ]:
rows = []
for f in (results / 'quast').glob('*/report.tsv'):
    try:
        df = pd.read_csv(f, sep='\t', index_col=0)
        sid = df.columns[0]
        get = lambda k: df.loc[k, sid] if k in df.index else None
        nc = get('# contigs'); tl = get('Total length'); n50 = get('N50')
        issues = []
        if nc  and nc  > 500:      issues.append(f'high_contigs({nc})')
        if tl  and tl  < 1_000_000: issues.append(f'small_assembly({tl})')
        if n50 and n50 < 10_000:   issues.append(f'low_N50({n50})')
        rows.append({
            'sample_id':        sid,
            'num_contigs':      nc,
            'assembly_length':  tl,
            'n50':              n50,
            'gc_percent':       get('GC (%)'),
            'assembly_quality': 'Pass' if not issues else 'Fail: ' + ', '.join(issues),
        })
    except Exception as e:
        print(f'  QUAST skip {f}: {e}')

pd.DataFrame(rows).to_sql('assembly_stats', con, if_exists='replace', index=False)
print(f'assembly_stats: {len(rows)} rows')

## Genome completeness — BUSCO + CheckM2

In [ ]:
busco = {}
for f in (results / 'busco').glob('*/short_summary.*.txt'):
    sid = f.parent.name.replace('_busco', '')
    try:
        txt = f.read_text()
        m = re.search(r'C:([0-9.]+)%\[S:([0-9.]+)%,D:([0-9.]+)%\],F:([0-9.]+)%,M:([0-9.]+)%', txt)
        if m:
            busco[sid] = dict(busco_complete=float(m[1]), busco_duplicated=float(m[3]),
                              busco_fragmented=float(m[4]), busco_missing=float(m[5]))
    except Exception as e:
        print(f'  BUSCO skip {f}: {e}')

checkm2 = {}
checkm2_dir = results / 'checkm2'
if checkm2_dir.exists():
    for f in checkm2_dir.glob('*/quality_report.tsv'):
        sid = f.parent.name.replace('_checkm2', '')
        try:
            df = pd.read_csv(f, sep='\t')
            if not df.empty:
                r = df.iloc[0]
                comp = float(r.get('Completeness', 0)); cont = float(r.get('Contamination', 0))
                checkm2[sid] = dict(checkm2_completeness=comp, checkm2_contamination=cont,
                                    checkm2_model=str(r.get('Completeness_Model_Used', '')),
                                    checkm2_pass='Pass' if comp>=90 and cont<=5 else
                                                 'Medium' if comp>=50 and cont<=10 else 'Fail')
        except Exception as e:
            print(f'  CheckM2 skip {f}: {e}')
else:
    print('  checkm2 directory not found — skipping')

all_sids = set(busco) | set(checkm2)
rows = [{'sample_id': s, **busco.get(s, {}), **checkm2.get(s, {})} for s in all_sids]
pd.DataFrame(rows).to_sql('genome_quality', con, if_exists='replace', index=False)
print(f'genome_quality: {len(rows)} rows  (busco={len(busco)}, checkm2={len(checkm2)})')

## MLST

In [ ]:
rows = []
for f in (results / 'mlst').glob('*_mlst.tsv'):
    sid = f.stem.replace('_mlst', '')
    try:
        df = pd.read_csv(f, sep='\t', header=None)
        if not df.empty:
            rows.append({'sample_id': sid,
                         'mlst_scheme': str(df.iloc[0][1]) if len(df.columns)>1 else '-',
                         'mlst_st':     str(df.iloc[0][2]) if len(df.columns)>2 else '-'})
    except Exception as e:
        print(f'  MLST skip {f}: {e}')

pd.DataFrame(rows).to_sql('mlst', con, if_exists='replace', index=False)
print(f'mlst: {len(rows)} rows')

## Serotyping — ECTyper

In [ ]:
rows = []
for f in (results / 'ectyper').glob('*_ectyper.tsv'):
    sid = f.stem.replace('_ectyper', '')
    try:
        df = pd.read_csv(f, sep='\t', dtype=str).fillna('-')
        if not df.empty:
            r = df.iloc[0]
            rows.append({'sample_id':   sid,
                         'o_type':      r.get('O-type', '-'),
                         'h_type':      r.get('H-type', '-'),
                         'serotype':    r.get('Serotype', '-'),
                         'ectyper_qc':  r.get('QC', '-'),
                         'pathotype':   r.get('Pathotype', '-'),
                         'stx_subtypes':r.get('StxSubtypes', '-')})
    except Exception as e:
        print(f'  ECTyper skip {f}: {e}')

pd.DataFrame(rows).to_sql('serotyping', con, if_exists='replace', index=False)
print(f'serotyping: {len(rows)} rows')

## AMR — AMRFinder (summary + individual gene hits)

In [ ]:
summary_rows = []
gene_rows    = []

for f in (results / 'amrfinder').glob('*_amr.tsv'):
    sid = f.stem.replace('_amr', '')
    try:
        df = pd.read_csv(f, sep='\t')
        if df.empty:
            summary_rows.append({'sample_id': sid, 'num_amr_genes': 0,
                                 'num_point_mutations': 0, 'amr_classes': '-',
                                 'mdr_status': 'No', 'top_amr_genes': '-'})
            continue
        genes   = df[df['Element type']=='AMR']   if 'Element type' in df.columns else df
        points  = df[df['Element type']=='POINT']  if 'Element type' in df.columns else pd.DataFrame()
        classes = sorted(df['Class'].dropna().unique().tolist()) if 'Class' in df.columns else []
        top     = [g for g,_ in Counter(df['Gene symbol'].dropna()).most_common(5)] if 'Gene symbol' in df.columns else []
        summary_rows.append({'sample_id':          sid,
                             'num_amr_genes':       len(genes),
                             'num_point_mutations': len(points),
                             'amr_classes':         ', '.join(classes) if classes else '-',
                             'mdr_status':          'Yes' if len(classes)>=3 else 'No',
                             'top_amr_genes':       ', '.join(top) if top else '-'})
        # Individual gene hits
        for _, row in df.iterrows():
            gene_rows.append({'sample_id':    sid,
                              'gene_symbol':  row.get('Gene symbol'),
                              'element_type': row.get('Element type'),
                              'amr_class':    row.get('Class'),
                              'subclass':     row.get('Subclass'),
                              'method':       row.get('Method'),
                              'pct_coverage': row.get('% Coverage of reference sequence'),
                              'pct_identity': row.get('% Identity to reference sequence')})
    except Exception as e:
        print(f'  AMRFinder skip {f}: {e}')

pd.DataFrame(summary_rows).to_sql('amr_summary', con, if_exists='replace', index=False)
pd.DataFrame(gene_rows).to_sql('amr_genes',    con, if_exists='replace', index=False)
print(f'amr_summary: {len(summary_rows)} rows | amr_genes: {len(gene_rows)} rows')

## Plasmids — MOBsuite

In [ ]:
rows = []
for sample_dir in (results / 'mobsuite').glob('*_mobsuite'):
    sid = sample_dir.name.replace('_mobsuite', '')
    try:
        plasmid_files = list(sample_dir.glob('plasmid_*_typing.txt'))
        if not plasmid_files:
            rows.append({'sample_id': sid, 'num_plasmids': 0, 'inc_groups': '-', 'mob_types': '-'})
            continue
        inc, mob = [], []
        for pf in plasmid_files:
            df = pd.read_csv(pf, sep='\t')
            if 'rep_type(s)' in df.columns:
                for v in df['rep_type(s)'].dropna():
                    for r in str(v).split(','):
                        r = r.strip()
                        if r.startswith('Inc'): inc.append(r)
            if 'predicted_mobility' in df.columns:
                mob += [str(m) for m in df['predicted_mobility'].dropna()]
        rows.append({'sample_id':   sid,
                     'num_plasmids':len(plasmid_files),
                     'inc_groups':  ', '.join(sorted(set(inc))) if inc else '-',
                     'mob_types':   ', '.join(sorted(set(mob))) if mob else '-'})
    except Exception as e:
        print(f'  MOBsuite skip {sample_dir}: {e}')

pd.DataFrame(rows).to_sql('plasmids', con, if_exists='replace', index=False)
print(f'plasmids: {len(rows)} rows')

## Prophages — VIBRANT

In [ ]:
rows = []
for f in (results / 'vibrant').glob('*/VIBRANT_*/VIBRANT_results_*/VIBRANT_genome_quality_*.tsv'):
    sid = f.parents[2].name.replace('_vibrant', '')
    try:
        df = pd.read_csv(f, sep='\t')
        rows.append({'sample_id':     sid,
                     'num_prophages': len(df),
                     'num_lytic':     int((df['type']=='lytic').sum())    if 'type' in df.columns else 0,
                     'num_lysogenic': int((df['type']=='lysogenic').sum()) if 'type' in df.columns else 0})
    except Exception as e:
        print(f'  VIBRANT skip {f}: {e}')

pd.DataFrame(rows).to_sql('prophages', con, if_exists='replace', index=False)
print(f'prophages: {len(rows)} rows')

## Defense systems — DefenseFinder + PADLOC

In [ ]:
df_rows = []
for f in (results / 'defensefinder').glob('*_defense_finder_systems.tsv'):
    sid = f.name.replace('_defense_finder_systems.tsv', '')
    try:
        df = pd.read_csv(f, sep='\t')
    except pd.errors.EmptyDataError:
        df = pd.DataFrame()
    act = df['activity'].astype(str).str.lower() if not df.empty and 'activity' in df.columns else pd.Series(dtype=str)
    def_df  = df[act=='defense']                           if not act.empty else df
    anti_df = df[act.isin(['anti-defense','antidefense'])] if not act.empty else pd.DataFrame()
    types      = sorted(set(def_df['type'].astype(str)))  if not def_df.empty  else []
    anti_types = sorted(set(anti_df['type'].astype(str))) if not anti_df.empty else []
    df_rows.append({'sample_id':              sid,
                    'defense_system_count':   len(def_df),
                    'defense_system_types':   ';'.join(types)      if types      else 'none',
                    'defense_has_crispr_cas': 'Yes' if any(t.lower().startswith('cas') for t in types) else 'No',
                    'antidefense_count':      len(anti_df),
                    'antidefense_types':      ';'.join(anti_types) if anti_types else 'none'})

pd.DataFrame(df_rows).to_sql('defense_systems', con, if_exists='replace', index=False)
print(f'defense_systems: {len(df_rows)} rows')

padloc_rows = []
for f in (results / 'padloc').glob('*_padloc.csv'):
    sid = f.name.replace('_padloc.csv', '')
    try:
        df = pd.read_csv(f)
    except pd.errors.EmptyDataError:
        padloc_rows.append({'sample_id': sid, 'padloc_system_count': 0, 'padloc_system_types': 'none'})
        continue
    sys_col = next((c for c in ['system','system_name','target.name'] if c in df.columns), None)
    systems = sorted(set(df[sys_col].dropna().astype(str))) if sys_col else []
    padloc_rows.append({'sample_id':           sid,
                        'padloc_system_count': len(df),
                        'padloc_system_types': ';'.join(systems) if systems else 'none'})

pd.DataFrame(padloc_rows).to_sql('padloc', con, if_exists='replace', index=False)
print(f'padloc: {len(padloc_rows)} rows')

## CRISPR — MinCED

In [ ]:
rows = []
for sample_dir in (results / 'crispr' / 'minced').iterdir():
    if not sample_dir.is_dir(): continue
    sid = sample_dir.name.replace('_crispr', '')
    files = list(sample_dir.glob('*_crispr_summary.csv'))
    if not files:
        rows.append({'sample_id': sid, 'crispr_array_count': 0, 'crispr_spacer_count': 0, 'crispr_repeat_avg': None})
        continue
    try:
        df = pd.read_csv(files[0])
        rows.append({'sample_id':          sid,
                     'crispr_array_count': len(df),
                     'crispr_spacer_count':int(df['spacers'].sum()) if 'spacers' in df.columns else 0,
                     'crispr_repeat_avg':  round(float(df['repeats'].mean()), 1) if 'repeats' in df.columns and not df.empty else None})
    except Exception as e:
        print(f'  MinCED skip {sample_dir}: {e}')

pd.DataFrame(rows).to_sql('crispr', con, if_exists='replace', index=False)
print(f'crispr: {len(rows)} rows')

## Virulence factors — VFDB (ABRicate)

In [ ]:
rows = []
for f in (results / 'abricate').glob('*_vfdb.tsv'):
    sid = f.stem.replace('_vfdb', '')
    try:
        df = pd.read_csv(f, sep='\t')
        df.columns = df.columns.str.replace('^#', '', regex=True)
        if df.empty:
            rows.append({'sample_id': sid, 'vf_gene_count': 0, 'vf_genes': '',
                         'vf_avg_identity': None, 'vf_avg_coverage': None})
            continue
        rows.append({'sample_id':       sid,
                     'vf_gene_count':   int(df['GENE'].nunique()),
                     'vf_total_hits':   len(df),
                     'vf_genes':        ', '.join(df['GENE'].unique().tolist()),
                     'vf_avg_identity': round(float(df['%IDENTITY'].mean()), 2) if '%IDENTITY' in df.columns else None,
                     'vf_avg_coverage': round(float(df['%COVERAGE'].mean()), 2) if '%COVERAGE' in df.columns else None})
    except Exception as e:
        print(f'  VFDB skip {f}: {e}')

pd.DataFrame(rows).to_sql('virulence_factors', con, if_exists='replace', index=False)
print(f'virulence_factors: {len(rows)} rows')

## Prophage identity — DIAMOND

In [ ]:
rows = []
DIAMOND_COLS = ['qseqid','sseqid','pident','length','mismatch','gapopen',
                'qstart','qend','sstart','send','evalue','bitscore']
for f in (results / 'diamond_prophage').glob('*_diamond_results.tsv'):
    sid = f.stem.replace('_diamond_results', '')
    try:
        df = pd.read_csv(f, sep='\t', header=None, names=DIAMOND_COLS)
        if df.empty:
            rows.append({'sample_id': sid, 'num_prophage_hits': 0,
                         'top_prophage_matches': '-', 'best_prophage_matches': '-'})
            continue
        top = df.nlargest(3, 'pident')
        top_str = ', '.join(f"{r.sseqid}({r.pident:.1f}%)" for _,r in top.iterrows())
        best = df.loc[df.groupby('qseqid')['bitscore'].idxmax()]
        best_str = ', '.join(f"{r.sseqid}({r.pident:.1f}%)" for _,r in
                             best.sort_values('bitscore', ascending=False).iterrows())
        rows.append({'sample_id':            sid,
                     'num_prophage_hits':    int(df['qseqid'].nunique()),
                     'top_prophage_matches': top_str,
                     'best_prophage_matches':best_str})
    except Exception as e:
        print(f'  DIAMOND skip {f}: {e}')

pd.DataFrame(rows).to_sql('diamond_prophage', con, if_exists='replace', index=False)
print(f'diamond_prophage: {len(rows)} rows')

## Annotation stats — Phanotate

In [ ]:
rows = []
phanotate_base = results / 'phanotate'
if phanotate_base.exists():
    for f in phanotate_base.glob('*_phanotate.gff'):
        sid = f.stem.replace('_phanotate', '')
        try:
            total_cds = hypo = 0
            for line in f.read_text().splitlines():
                if line.startswith('#'): continue
                parts = line.split('\t')
                if len(parts) >= 9 and parts[2] == 'CDS':
                    total_cds += 1
                    if 'hypothetical' in parts[8].lower(): hypo += 1
            rows.append({'sample_id':             sid,
                         'total_cds':             total_cds,
                         'hypothetical_proteins': hypo,
                         'hypothetical_pct':      round(hypo/total_cds*100, 2) if total_cds else 0})
        except Exception as e:
            print(f'  Phanotate skip {f}: {e}')

pd.DataFrame(rows).to_sql('phanotate', con, if_exists='replace', index=False)
print(f'phanotate: {len(rows)} rows')

## Prophage-encoded AMR

In [ ]:
rows = []
prophage_amr_base = results / 'prophage_amr'
if prophage_amr_base.exists():
    for f in prophage_amr_base.glob('*_prophage_amr.tsv'):
        sid = f.stem.replace('_prophage_amr', '')
        try:
            df = pd.read_csv(f, sep='\t', comment='#')
            if df.empty or 'amr_gene' not in df.columns:
                rows.append({'sample_id': sid, 'prophage_amr_gene_count': 0,
                             'prophage_amr_genes': '', 'prophage_amr_classes': '', 'has_prophage_amr': 0})
                continue
            retained = df[df['excluded']==False] if 'excluded' in df.columns else df
            genes   = [g for g in retained['amr_gene'].unique()  if pd.notna(g)]
            classes = [c for c in retained['amr_class'].unique() if pd.notna(c)] if 'amr_class' in retained.columns else []
            rows.append({'sample_id':              sid,
                         'prophage_amr_gene_count':len(genes),
                         'prophage_amr_genes':     ', '.join(genes),
                         'prophage_amr_classes':   ', '.join(classes),
                         'has_prophage_amr':       int(len(genes)>0)})
        except Exception as e:
            print(f'  Prophage-AMR skip {f}: {e}')

pd.DataFrame(rows).to_sql('prophage_amr', con, if_exists='replace', index=False)
print(f'prophage_amr: {len(rows)} rows')

## Serotyping (Salmonella) — SISTR

In [ ]:
rows = []
for f in (results / 'sistr').glob('*_sistr.tsv') if (results / 'sistr').exists() else []:
    sid = f.stem.replace('_sistr', '')
    try:
        df = pd.read_csv(f, sep='\t')
        if not df.empty:
            rows.append({'sample_id': sid,
                         'serovar':   df.iloc[0].get('serovar', '-'),
                         'serogroup': df.iloc[0].get('serogroup', '-'),
                         'h1':        df.iloc[0].get('h1', '-'),
                         'h2':        df.iloc[0].get('h2', '-')})
    except Exception as e:
        print(f'  SISTR skip {f}: {e}')

pd.DataFrame(rows).to_sql('sistr', con, if_exists='replace', index=False)
print(f'sistr: {len(rows)} rows')

## geNomad prophage ICTV taxonomy

In [ ]:
rows = []
genomad_base = results / 'genomad_prophage'
if genomad_base.exists():
    for sample_dir in genomad_base.iterdir():
        if not sample_dir.is_dir(): continue
        sid = sample_dir.name.replace('_genomad', '')
        # Try both common nesting patterns
        candidates = [
            sample_dir / f'{sid}_genomad' / f'{sid}_summary' / f'{sid}_virus_summary.tsv',
            sample_dir / f'{sid}_summary' / f'{sid}_virus_summary.tsv',
        ]
        tsv = next((p for p in candidates if p.exists()), None)
        if not tsv: continue
        try:
            df = pd.read_csv(tsv, sep='\t')
            if df.empty: continue
            tax_col = next((c for c in df.columns if 'taxonomy' in c.lower()), None)
            families = []
            if tax_col:
                for t in df[tax_col].dropna():
                    parts = str(t).split(';')
                    if len(parts) >= 7: families.append(parts[6].strip())
            rows.append({'sample_id':       sid,
                         'num_viruses':     len(df),
                         'ictv_families':   '; '.join(sorted(set(families))) if families else '-'})
        except Exception as e:
            print(f'  geNomad skip {sample_dir}: {e}')

pd.DataFrame(rows).to_sql('genomad_ictv', con, if_exists='replace', index=False)
print(f'genomad_ictv: {len(rows)} rows')

In [ ]:
tables = ['assembly_stats','genome_quality','mlst','serotyping','amr_summary',
          'plasmids','prophages','defense_systems','padloc','crispr',
          'virulence_factors','diamond_prophage','phanotate','prophage_amr',
          'sistr','genomad_ictv']

all_ids = set()
for t in tables:
    try:
        all_ids |= set(pd.read_sql(f'SELECT DISTINCT sample_id FROM {t}', con)['sample_id'])
    except Exception:
        pass

pd.DataFrame({'sample_id': sorted(all_ids)}).to_sql('samples', con, if_exists='replace', index=False)
print(f'samples (master): {len(all_ids)} unique sample_ids')

## Master samples table
Union of all `sample_id`s seen across every table.

## Verify — table row counts

In [ ]:
cur = con.cursor()
cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
for (tbl,) in cur.fetchall():
    n = pd.read_sql(f'SELECT COUNT(*) AS n FROM {tbl}', con).iloc[0]['n']
    print(f'  {tbl:<25} {n:>6} rows')
con.close()
print(f'\nDatabase saved to {DB_PATH}')